# Skin Disease — Multimodal Fusion (Image + Text)

Builds on the two earlier notebooks:

| From | What is reused here |
|---|---|
| `Skin_Disease_CK_IMAGE.ipynb` | colour normalisation (brightness / contrast / saturation → dataset median) and the geometric augmentation used to balance classes |
| `Skin_Disease_CK_TEXT.ipynb` | UNK cleaning, clinical encoding (TRUE=1, FALSE=0, UNK=2, region one-hot), the 5 patient-grouped folds, and the best text model: **Logistic Regression + Custom SMOTE** |

### Models in this notebook
| | Model | Input | Type |
|---|---|---|---|
| **A** | MobileNetV2 CNN (ImageNet transfer learning) | lesion photo | image only |
| **B** | Logistic Regression + Custom SMOTE | 21 clinical features | text only |
| **C** | Late fusion: `w·P(A) + (1−w)·P(B)` | both | decision-level fusion |
| **D** | Feature-fusion network: CNN image features + clinical MLP features → joint classifier | both | feature-level fusion |

A and B are trained **separately**. C combines their outputs; D combines their internal features and is trained end-to-end.
All four are tested on the **same unseen patients**, then the image evidence is explained with **Grad-CAM**.

### How to run (Google Colab)
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` — about 20–25 minutes
3. Allow Google Drive access — models and results are saved to `MyDrive/skin_fusion/`

In [ ]:
# ===================== CONFIGURATION =====================
SEED          = 42
IMG_SIZE      = 224      # MobileNetV2 input size
WORK_SIZE     = 320      # photos are normalised / augmented at this size, then resized to IMG_SIZE
BATCH_SIZE    = 32
SAVE_TO_DRIVE = True
MAX_AUG_PER_IMG = 10     # same balancing rule as the IMAGE notebook

# Model A - image CNN (stage 1: frozen backbone, stage 2: fine-tune top of backbone)
IMG_HEAD_EPOCHS, IMG_HEAD_LR         = 10, 1e-3
IMG_FINETUNE_EPOCHS, IMG_FINETUNE_LR = 25, 5e-5
FINE_TUNE_FROM = "block_13_expand"

# Model D - feature-fusion network (stage 1: image branch frozen, stage 2: fine-tune)
FUSION_HEAD_EPOCHS, FUSION_HEAD_LR         = 20, 1e-3
FUSION_FINETUNE_EPOCHS, FUSION_FINETUNE_LR = 15, 1e-5

KAGGLE_DATASET = "mahdavi1202/skin-cancer"    # PAD-UFES-20

In [ ]:
!pip install -q imbalanced-learn

import os, glob, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import tensorflow as tf
import keras
from keras import layers

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, roc_auc_score, confusion_matrix)
from sklearn.exceptions import ConvergenceWarning
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore", category=ConvergenceWarning)
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE  ->  Runtime > Change runtime type > T4 GPU")

OUT_DIR, CKPT_DIR = "outputs", "checkpoints"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/skin_fusion"
    except ImportError:
        print("Not in Colab - saving to ./outputs")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
print("Saving results to:", OUT_DIR)

## 1. Load the dataset (photos + `metadata.csv`)

Same cleaning as the TEXT notebook: rows with `UNK` in itch / hurt / bleed / elevation are dropped (19 rows → 2,279 left).
`grew` and `changed` keep `UNK` because ~400 rows have it.

In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download(KAGGLE_DATASET)

df = pd.read_csv(os.path.join(DATA_ROOT, "metadata.csv"))
path_map = {os.path.basename(p): p for p in glob.glob(os.path.join(DATA_ROOT, "**", "*.png"), recursive=True)}
df["path"] = df["img_id"].map(path_map)
print(f"metadata rows: {len(df)} | images found: {df['path'].notna().sum()}")

SYMPTOMS = ["itch", "grew", "hurt", "changed", "bleed", "elevation"]
for c in SYMPTOMS + ["region"]:
    df[c] = df[c].astype(str).str.upper().str.strip()

keep = df[["itch", "hurt", "bleed", "elevation"]].ne("UNK").all(axis=1) & df["path"].notna()
data = df.loc[keep].reset_index(drop=True)

CLASS_NAMES = sorted(data["diagnostic"].unique())             # ['ACK','BCC','MEL','NEV','SCC','SEK']
CLASS_FULL = {"ACK": "Actinic Keratosis", "BCC": "Basal Cell Carcinoma", "MEL": "Melanoma",
              "NEV": "Nevus", "SCC": "Squamous Cell Carcinoma", "SEK": "Seborrheic Keratosis"}
CLASS_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
y_all = data["diagnostic"].map(CLASS_IDX).to_numpy()
print(f"rows used: {len(data)}")
data["diagnostic"].value_counts()

## 2. Patient-level split (same folds as the TEXT notebook's `splits.csv`)

`StratifiedGroupKFold(5, shuffle=True, random_state=42)` grouped by `patient_id` — identical to the TEXT notebook.
fold 0 → **test**, fold 1 → **validation**, folds 2–4 → **training** (≈ 60 / 20 / 20).
No patient appears in two splits, so the test score is honest.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
data["fold"] = -1
for k, (_, fold_idx) in enumerate(sgkf.split(np.zeros(len(data)), data["diagnostic"], groups=data["patient_id"])):
    data.loc[fold_idx, "fold"] = k
data["split"] = data["fold"].map(lambda k: "test" if k == 0 else ("val" if k == 1 else "train"))

tr_idx = np.where(data["split"] == "train")[0]
va_idx = np.where(data["split"] == "val")[0]
te_idx = np.where(data["split"] == "test")[0]

pats = {s: set(data.loc[data["split"] == s, "patient_id"]) for s in ("train", "val", "test")}
assert not (pats["train"] & pats["val"]) and not (pats["train"] & pats["test"]) and not (pats["val"] & pats["test"])
print(f"train {len(tr_idx)} | val {len(va_idx)} | test {len(te_idx)} images - no patient shared between splits")
data[["img_id", "patient_id", "diagnostic", "fold", "split"]].to_csv(f"{OUT_DIR}/splits.csv", index=False)
pd.crosstab(data["split"], data["diagnostic"], margins=True).loc[["train", "val", "test", "All"]]

## 3. Text (clinical) features — same encoding as the TEXT notebook

`age` (number) + 6 symptoms (`TRUE`=1, `FALSE`=0, `UNK`=2) + 14 body-region one-hot columns = **21 features** per lesion.

In [ ]:
REGIONS = sorted(data["region"].unique())

def encode_clinical(frame):
    """age + symptoms (TRUE=1, FALSE=0, UNK=2) + region one-hot, in the TEXT notebook's column order."""
    X = pd.DataFrame({"age": frame["age"].astype(float).to_numpy()})
    for c in SYMPTOMS:
        X[c] = (frame[c].astype(str).str.upper().str.strip()
                .map({"TRUE": 1, "FALSE": 0, "UNK": 2}).fillna(2).astype(float).to_numpy())
    region = frame["region"].astype(str).str.upper().str.strip().to_numpy()
    for r in REGIONS:
        X[f"region_{r}"] = (region == r).astype(float)
    return X

X_meta_df = encode_clinical(data)
X_meta = X_meta_df.to_numpy("float32")
TEXT_FEATURES = list(X_meta_df.columns)
print(len(TEXT_FEATURES), "features:", TEXT_FEATURES)
X_meta_df.head()

## 4. Image preprocessing — same steps as the IMAGE notebook

1. **Colour normalisation:** each photo's brightness (LAB L mean), contrast (L std) and saturation (HSV S mean)
   are moved to the median of the training photos. This removes lighting / camera differences between photos.
2. **Class balancing by augmentation:** flips, 90° rotations, ±30° rotation, zoom, shift and shear (no colour change),
   with the IMAGE notebook's rule: every class is topped up towards the largest class, at most 10 new images per photo.

**Difference from the IMAGE notebook:** augmentation is done **after** the patient split and **only for training patients**.
(In the IMAGE notebook all 2,298 photos were augmented before splitting — training on that folder would put
augmented copies of test photos into training and inflate the score.)

In [ ]:
def get_stats(img_bgr):
    L = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)[..., 0].astype(np.float32)
    S = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[..., 1].astype(np.float32)
    return L.mean(), L.std(), S.mean()

def normalize_image(img_bgr, t_bright, t_contr, t_sat, max_scale=2.0):
    # 1) saturation: scale S so its mean = target
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    s_scale = np.clip(t_sat / (hsv[..., 1].mean() + 1e-6), 1 / max_scale, max_scale)
    hsv[..., 1] = np.clip(hsv[..., 1] * s_scale, 0, 255)
    img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    # 2) brightness + contrast: shift/scale L so mean = target, std = target
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
    L = lab[..., 0]
    c_scale = np.clip(t_contr / (L.std() + 1e-6), 1 / max_scale, max_scale)
    lab[..., 0] = np.clip((L - L.mean()) * c_scale + t_bright, 0, 255)
    return cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)

def augment(img):
    """Geometric augmentation from the IMAGE notebook (no colour change). RGB in, IMG_SIZE x IMG_SIZE RGB out."""
    if random.random() < 0.5: img = np.fliplr(img)
    if random.random() < 0.5: img = np.flipud(img)
    img = np.ascontiguousarray(np.rot90(img, k=random.randint(0, 3)))
    h, w = img.shape[:2]
    angle = random.uniform(-30, 30)
    scale = random.uniform(1.0, 1.2)                       # zoom in only
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    M[0, 2] += random.uniform(-0.08, 0.08) * w             # shift x
    M[1, 2] += random.uniform(-0.08, 0.08) * h             # shift y
    M[0, 1] += np.tan(np.deg2rad(random.uniform(-8, 8)))   # shear
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

def read_work(path):
    """Read a photo (BGR) and shrink it to WORK_SIZE x WORK_SIZE."""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    return cv2.resize(img, (WORK_SIZE, WORK_SIZE), interpolation=cv2.INTER_AREA)

def preprocess_photo(path):
    """Full image pipeline for one photo: read -> colour-normalise -> RGB -> IMG_SIZE. Used for new photos too."""
    bgr = normalize_image(read_work(path), T_BRIGHT, T_CONTR, T_SAT)
    return cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

In [ ]:
# 1) read every photo once
with ThreadPoolExecutor(max_workers=8) as pool:
    work = list(tqdm(pool.map(read_work, data["path"]), total=len(data), desc="Reading photos"))

# 2) normalisation targets = median of the TRAINING photos
stats = np.array([get_stats(im) for im in work])
T_BRIGHT, T_CONTR, T_SAT = [float(v) for v in np.median(stats[tr_idx], axis=0)]
print(f"Targets -> brightness {T_BRIGHT:.1f} | contrast {T_CONTR:.1f} | saturation {T_SAT:.1f}")

# 3) normalise every photo (kept in RGB at WORK_SIZE for augmentation) and make the 224x224 model inputs
examples = {i: cv2.cvtColor(work[i], cv2.COLOR_BGR2RGB) for i in tr_idx[:3]}
norm_work = [cv2.cvtColor(normalize_image(im, T_BRIGHT, T_CONTR, T_SAT), cv2.COLOR_BGR2RGB)
             for im in tqdm(work, desc="Normalising")]
del work
X_img = np.stack([cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA) for im in norm_work])

after = np.array([get_stats(cv2.cvtColor(im, cv2.COLOR_RGB2BGR)) for im in norm_work])
print(pd.DataFrame({"std before": stats.std(0), "std after": after.std(0)},
                   index=["brightness", "contrast", "saturation"]).round(2))
print("model images:", X_img.shape)

In [ ]:
# 4) balance the TRAINING patients' photos with the IMAGE notebook's augmentation rule
random.seed(SEED); np.random.seed(SEED)
y_tr = y_all[tr_idx]
target = np.bincount(y_tr, minlength=NUM_CLASSES).max()
aug_imgs, aug_src = [], []
for c in range(NUM_CLASSES):
    src = tr_idx[y_tr == c]
    need = max(min(target - len(src), len(src) * MAX_AUG_PER_IMG), 0)
    for i in range(need):
        row = src[i % len(src)]
        aug_imgs.append(augment(norm_work[row]))
        aug_src.append(row)
aug_src = np.array(aug_src, dtype=int)

# training set = original training photos + their augmented copies.
# train_rows says which patient row each training image came from (gives its label and clinical features).
train_img = np.concatenate([X_img[tr_idx], np.stack(aug_imgs)])
train_rows = np.concatenate([tr_idx, aug_src])
y_train = y_all[train_rows]
print(pd.DataFrame({"before": np.bincount(y_tr, minlength=NUM_CLASSES),
                    "after augmentation": np.bincount(y_train, minlength=NUM_CLASSES)}, index=CLASS_NAMES))

fig, axes = plt.subplots(len(examples), 4, figsize=(12, 3.2 * len(examples)), squeeze=False)
for r, i in enumerate(examples):
    random.seed(r)
    shown = [examples[i], norm_work[i], augment(norm_work[i]), augment(norm_work[i])]
    for ax, im, t in zip(axes[r], shown, ["original", "normalised", "augmented", "augmented"]):
        ax.imshow(im); ax.set_title(f"{t} ({CLASS_NAMES[y_all[i]]})"); ax.axis("off")
plt.tight_layout(); plt.show()
del norm_work, aug_imgs

### Training and evaluation helpers

* Loss: categorical cross-entropy with class weights (small correction — augmentation already balanced most classes).
* Model selection: keep the epoch with the best **validation macro-F1** (every class counts equally).
* Test report: accuracy, balanced accuracy, macro precision / recall / F1, weighted F1, macro ROC-AUC,
  per-class precision, recall (sensitivity), F1, **false-positive rate**, and confusion matrices.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
ONLINE_AUG = [layers.RandomFlip("horizontal_and_vertical"),
              layers.RandomRotation(0.1, fill_mode="reflect")]     # light extra variety every epoch

def make_dataset(images, meta, labels, inputs, training=False):
    """inputs = 'image' (Model A) or 'both' (Model D)."""
    x = images if inputs == "image" else {"image": images, "meta": meta}
    y = keras.utils.to_categorical(labels, NUM_CLASSES).astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(labels), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)

    def prepare(x, y):
        img = tf.cast(x if inputs == "image" else x["image"], tf.float32)
        if training:
            for layer in ONLINE_AUG:
                img = layer(img, training=True)
        return (img if inputs == "image" else {"image": img, "meta": x["meta"]}), y

    return ds.map(prepare, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
print("class weights:", {CLASS_NAMES[i]: round(w, 2) for i, w in CLASS_WEIGHT.items()})

HEAD_PREFIXES = ("img_", "meta_", "fusion_")    # all layers we add start with one of these

def is_backbone(layer):
    return not isinstance(layer, keras.layers.InputLayer) and not layer.name.startswith(HEAD_PREFIXES)

def set_backbone_trainable(model, from_layer=None):
    """Freeze MobileNetV2; if from_layer is given, unfreeze from that layer to the end (BatchNorm stays frozen)."""
    assert from_layer is None or from_layer in [l.name for l in model.layers], f"{from_layer} not found"
    unfreeze = False
    for layer in model.layers:
        if not is_backbone(layer):
            continue
        if layer.name == from_layer:
            unfreeze = True
        layer.trainable = unfreeze and not isinstance(layer, layers.BatchNormalization)

def count_params(model):
    t = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    f = sum(int(np.prod(w.shape)) for w in model.non_trainable_weights)
    print(f"{model.name}: {t:,} trainable / {f:,} frozen parameters")

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
                  metrics=["accuracy", keras.metrics.F1Score(average="macro", name="macro_f1")])

def fit_stage(model, train_ds, val_ds, epochs, ckpt_path, patience, best_so_far=None):
    """Train one stage; the best weights by val macro-F1 (over all stages) are kept in ckpt_path."""
    callbacks = [
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_macro_f1", mode="max", save_best_only=True,
                                        save_weights_only=True, initial_value_threshold=best_so_far),
        keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=patience),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ]
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=CLASS_WEIGHT,
                     callbacks=callbacks, verbose=2, shuffle=False)   # shuffling is done in make_dataset
    best = max(hist.history["val_macro_f1"])
    return hist.history, best if best_so_far is None else max(best, best_so_far)

def load_best(model, ckpt_path):
    """Restore the best epoch's weights. The checkpoint may come from stage 1, whose optimizer had fewer
    variables than stage 2's - Keras then skips the optimizer state (not needed after training) and warns."""
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Skipping variable loading for optimizer")
        model.load_weights(ckpt_path)

def plot_history(histories, title, key):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, m in zip(axes, ["loss", "accuracy", "macro_f1"]):
        ax.plot(sum((h[m] for h in histories), []), label="train")
        ax.plot(sum((h["val_" + m] for h in histories), []), label="validation")
        if len(histories) > 1:
            ax.axvline(len(histories[0][m]) - 0.5, ls="--", c="gray", label="fine-tuning starts")
        ax.set_title(m); ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend()
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/history_{key}.png", dpi=150, bbox_inches="tight"); plt.show()

RESULTS, PER_CLASS, VAL_PROBS, TEST_PROBS = {}, {}, {}, {}

def evaluate(title, key, probs):
    """Test-set report for one model (summary + per-class table + confusion matrices). Saved to OUT_DIR."""
    y_true, y_pred = y_all[te_idx], probs.argmax(axis=1)
    labels = list(range(NUM_CLASSES))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fp = cm.sum(axis=0) - np.diag(cm)
    tn = cm.sum() - cm.sum(axis=0) - cm.sum(axis=1) + np.diag(cm)
    try:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro", labels=labels)
    except ValueError:
        auc = float("nan")
    RESULTS[title] = {"Accuracy": accuracy_score(y_true, y_pred),
                      "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
                      "Macro precision": p.mean(), "Macro recall": r.mean(), "Macro F1": f.mean(),
                      "Weighted F1": f1_score(y_true, y_pred, average="weighted"), "Macro AUC": auc}
    per_class = pd.DataFrame({"precision": p, "recall (sensitivity)": r, "F1": f,
                              "false-positive rate": fp / np.maximum(fp + tn, 1), "support": s}, index=CLASS_NAMES)
    PER_CLASS[title] = per_class
    per_class.to_csv(f"{OUT_DIR}/per_class_{key}.csv")
    preds = pd.DataFrame(probs, columns=CLASS_NAMES)
    preds.insert(0, "img_id", data["img_id"].to_numpy()[te_idx])
    preds.insert(1, "true", [CLASS_NAMES[i] for i in y_true])
    preds.insert(2, "predicted", [CLASS_NAMES[i] for i in y_pred])
    preds.to_csv(f"{OUT_DIR}/test_predictions_{key}.csv", index=False)

    print(f"===== {title} - TEST SET ({len(te_idx)} images) =====")
    display(pd.DataFrame([RESULTS[title]], index=[title]).round(3))
    display(per_class.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title("Confusion matrix (counts)")
    sns.heatmap(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), annot=True, fmt=".2f", cmap="Blues",
                vmin=0, vmax=1, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_title("Row-normalised (diagonal = recall)")
    for ax in axes:
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/confusion_{key}.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Model A — Image model: MobileNetV2 (transfer learning)

```
photo (224×224×3) → scale to [-1, 1] → MobileNetV2 backbone (ImageNet weights, 1280 feature maps of 7×7)
      → global average pooling (1280) → Dropout → Dense 256 ReLU  [= image embedding]
      → Dropout → Dense 6 softmax
```
* **Why MobileNetV2:** lightweight (≈ 2.3 M backbone parameters, depth-wise separable convolutions) — the Phase I report asks for a MobileNet-family model that runs on ordinary computers.
* **Stage 1:** backbone frozen; only the new head learns (fast, avoids destroying ImageNet features).
* **Stage 2:** the last MobileNetV2 blocks (`block_13` onward) are unfrozen and fine-tuned with a small learning rate.

In [ ]:
def build_image_model():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1 / 127.5, offset=-1, name="img_preprocess")(inp)     # 0..255 -> -1..1
    backbone = keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                              input_tensor=x, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.GlobalAveragePooling2D(name="img_gap")(backbone.output)
    x = layers.Dropout(0.3, name="img_dropout1")(x)
    x = layers.Dense(256, activation="relu", name="img_embedding")(x)
    x = layers.Dropout(0.3, name="img_dropout2")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="img_output")(x)
    return keras.Model(inp, out, name="image_model")

image_model = build_image_model()
img_train_ds = make_dataset(train_img, None, y_train, "image", training=True)
img_val_ds = make_dataset(X_img[va_idx], None, y_all[va_idx], "image")
img_test_ds = make_dataset(X_img[te_idx], None, y_all[te_idx], "image")

ckpt = f"{CKPT_DIR}/image_model.weights.h5"
print("--- Stage 1: head only (backbone frozen) ---")
set_backbone_trainable(image_model)
compile_model(image_model, IMG_HEAD_LR); count_params(image_model)
h1, best = fit_stage(image_model, img_train_ds, img_val_ds, IMG_HEAD_EPOCHS, ckpt, patience=4)

print(f"--- Stage 2: fine-tune from {FINE_TUNE_FROM} ---")
set_backbone_trainable(image_model, FINE_TUNE_FROM)
compile_model(image_model, IMG_FINETUNE_LR); count_params(image_model)
h2, best = fit_stage(image_model, img_train_ds, img_val_ds, IMG_FINETUNE_EPOCHS, ckpt, patience=7, best_so_far=best)

load_best(image_model, ckpt)
image_model.save(f"{OUT_DIR}/image_model.keras")
print(f"best validation macro-F1 {best:.3f} -> saved image_model.keras")
plot_history([h1, h2], "Model A - Image (MobileNetV2)", "image")

In [ ]:
VAL_PROBS["image"] = image_model.predict(img_val_ds, verbose=0)
TEST_PROBS["image"] = image_model.predict(img_test_ds, verbose=0)
evaluate("A: Image (MobileNetV2)", "image", TEST_PROBS["image"])

## 6. Model B — Text model: Logistic Regression + Custom SMOTE (best model in the TEXT notebook)

In the TEXT notebook, Logistic Regression with the custom SMOTE profile had the best 5-fold macro-F1 (0.557),
ahead of XGBoost and Random Forest. It is retrained here on the training patients only.

* **SMOTE** creates synthetic minority-class rows by interpolating between neighbouring rows of the same class
  (only on the training split — validation / test rows are never touched).
* **Logistic Regression** learns one weight per clinical feature per class and outputs class probabilities with softmax.

In [ ]:
counts = pd.Series(y_all[tr_idx]).value_counts()
profile = {"BCC": 670, "ACK": 582, "NEV": 335, "SEK": 335, "SCC": 335, "MEL": 200}   # TEXT notebook's custom profile
total = sum(profile.values())
strategy = {CLASS_IDX[c]: max(int(t / total * len(tr_idx) * 1.35), int(counts.get(CLASS_IDX[c], 0)))
            for c, t in profile.items()}
smote = SMOTE(sampling_strategy=strategy, random_state=SEED, k_neighbors=min(3, counts.min() - 1))
X_res, y_res = smote.fit_resample(X_meta[tr_idx], y_all[tr_idx])
print(pd.DataFrame({"before SMOTE": np.bincount(y_all[tr_idx], minlength=NUM_CLASSES),
                    "after SMOTE": np.bincount(y_res, minlength=NUM_CLASSES)}, index=CLASS_NAMES))

text_model = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_res, y_res)
joblib.dump(text_model, f"{OUT_DIR}/text_model.joblib")

VAL_PROBS["text"] = text_model.predict_proba(X_meta[va_idx])
TEST_PROBS["text"] = text_model.predict_proba(X_meta[te_idx])
evaluate("B: Text (LogReg + SMOTE)", "text", TEST_PROBS["text"])

coef = pd.DataFrame(text_model.coef_.T, index=TEXT_FEATURES, columns=CLASS_NAMES)
plt.figure(figsize=(8, 7)); sns.heatmap(coef, cmap="coolwarm", center=0, annot=True, fmt=".1f", annot_kws={"size": 7})
plt.title("Model B: what each clinical feature pushes towards (LogReg weights)"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/text_model_weights.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Model C — Late fusion (decision level)

Each model votes with its probabilities; the votes are mixed:

`P_fused = w · P_image + (1 − w) · P_text`

`w` is picked on the **validation** patients (best macro-F1), then used once on the test patients.

In [ ]:
ws = np.round(np.arange(0, 1.0001, 0.05), 2)
val_f1 = [f1_score(y_all[va_idx], (w * VAL_PROBS["image"] + (1 - w) * VAL_PROBS["text"]).argmax(1), average="macro")
          for w in ws]
LATE_W = float(ws[int(np.argmax(val_f1))])
print(f"best image weight w = {LATE_W} (validation macro-F1 {max(val_f1):.3f})")

plt.figure(figsize=(7, 3.5)); plt.plot(ws, val_f1, marker="o"); plt.axvline(LATE_W, ls="--", c="gray")
plt.xlabel("w = weight of the image model  (0 = text only, 1 = image only)"); plt.ylabel("validation macro-F1")
plt.grid(alpha=0.3); plt.title("Late fusion weight")
plt.savefig(f"{OUT_DIR}/late_fusion_weight.png", dpi=150, bbox_inches="tight"); plt.show()

VAL_PROBS["late"] = LATE_W * VAL_PROBS["image"] + (1 - LATE_W) * VAL_PROBS["text"]
TEST_PROBS["late"] = LATE_W * TEST_PROBS["image"] + (1 - LATE_W) * TEST_PROBS["text"]
evaluate("C: Late fusion", "late_fusion", TEST_PROBS["late"])

## 8. Model D — Feature-level fusion network (the multimodal model)

```
photo ─► MobileNetV2 + Dense 256  (weights from trained Model A) ─► 256 image features ─┐
                                                                                         ├─► concat (288) ─► BatchNorm ─► Dense 128 ─► Dropout ─► softmax (6)
21 clinical features ─► standardise ─► Dense 64 ─► Dense 32 ───────► 32 clinical features ┘
```
Instead of mixing final answers (Model C), this network mixes **what each branch has learned** and can learn
combinations such as *"this colour pattern + bleeding + age 70 → BCC"*.

* **Stage 1:** image branch frozen (keeps Model A's knowledge); the clinical branch and fusion head learn.
* **Stage 2:** the whole network (backbone from `block_13`) is fine-tuned together with a very small learning rate.
* Each augmented training photo carries the clinical record of the patient it came from.

In [ ]:
img_branch = keras.models.load_model(f"{OUT_DIR}/image_model.keras", compile=False)
img_feat = img_branch.get_layer("img_embedding").output                       # 256-d

meta_in = keras.Input((X_meta.shape[1],), name="meta")
meta_norm = layers.Normalization(name="meta_norm")
meta_norm.adapt(X_meta[tr_idx])                                                # mean/std from training patients
t = meta_norm(meta_in)
t = layers.Dense(64, activation="relu", name="meta_dense1")(t)
t = layers.Dropout(0.3, name="meta_dropout1")(t)
txt_feat = layers.Dense(32, activation="relu", name="meta_embedding")(t)      # 32-d

x = layers.Concatenate(name="fusion_concat")([img_feat, txt_feat])
x = layers.BatchNormalization(name="fusion_bn")(x)
x = layers.Dense(128, activation="relu", name="fusion_dense")(x)
x = layers.Dropout(0.4, name="fusion_dropout")(x)
out = layers.Dense(NUM_CLASSES, activation="softmax", name="fusion_output")(x)
fusion_model = keras.Model({"image": img_branch.input, "meta": meta_in}, out, name="fusion_model")

def set_fusion_trainable(model, finetune):
    for layer in model.layers:
        if layer.name.startswith(("fusion_", "meta_")):
            layer.trainable = True
        elif layer.name.startswith("img_"):
            layer.trainable = finetune
    set_backbone_trainable(model, FINE_TUNE_FROM if finetune else None)

fus_train_ds = make_dataset(train_img, X_meta[train_rows], y_train, "both", training=True)
fus_val_ds = make_dataset(X_img[va_idx], X_meta[va_idx], y_all[va_idx], "both")
fus_test_ds = make_dataset(X_img[te_idx], X_meta[te_idx], y_all[te_idx], "both")

ckpt = f"{CKPT_DIR}/fusion_model.weights.h5"
print("--- Stage 1: clinical branch + fusion head (image branch frozen) ---")
set_fusion_trainable(fusion_model, finetune=False)
compile_model(fusion_model, FUSION_HEAD_LR); count_params(fusion_model)
f1h, best = fit_stage(fusion_model, fus_train_ds, fus_val_ds, FUSION_HEAD_EPOCHS, ckpt, patience=6)

print("--- Stage 2: fine-tune the whole fusion network ---")
set_fusion_trainable(fusion_model, finetune=True)
compile_model(fusion_model, FUSION_FINETUNE_LR); count_params(fusion_model)
f2h, best = fit_stage(fusion_model, fus_train_ds, fus_val_ds, FUSION_FINETUNE_EPOCHS, ckpt, patience=5, best_so_far=best)

load_best(fusion_model, ckpt)
fusion_model.save(f"{OUT_DIR}/fusion_model.keras")
print(f"best validation macro-F1 {best:.3f} -> saved fusion_model.keras")
plot_history([f1h, f2h], "Model D - Feature-level fusion", "fusion")

In [ ]:
VAL_PROBS["fusion"] = fusion_model.predict(fus_val_ds, verbose=0)
TEST_PROBS["fusion"] = fusion_model.predict(fus_test_ds, verbose=0)
evaluate("D: Feature fusion", "fusion", TEST_PROBS["fusion"])

## 9. Compare all models (same test patients)

In [ ]:
comparison = pd.DataFrame(RESULTS).T
comparison.to_csv(f"{OUT_DIR}/model_comparison.csv")
display(comparison.round(3))

cols = ["Accuracy", "Balanced accuracy", "Macro F1", "Macro AUC"]
ax = comparison[cols].plot.bar(figsize=(13, 4.5), rot=0, width=0.8)
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=7)
ax.set_ylim(0, 1.05); ax.set_ylabel("test score"); ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right", ncol=4); ax.set_title("Image vs Text vs Fusion")
plt.savefig(f"{OUT_DIR}/model_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

recall_table = pd.DataFrame({m: t["recall (sensitivity)"] for m, t in PER_CLASS.items()})
recall_table["test images"] = PER_CLASS[next(iter(PER_CLASS))]["support"]
recall_table.to_csv(f"{OUT_DIR}/per_class_recall_comparison.csv")
print("Per-class recall (sensitivity):")
display(recall_table.round(3))
print("Best model by test macro-F1:", comparison["Macro F1"].idxmax())

## 10. Grad-CAM — which part of the photo drove the prediction

Grad-CAM takes the gradient of the predicted class score with respect to MobileNetV2's last feature maps (`out_relu`, 7×7×1280),
averages it per feature map to get an importance weight, and sums the weighted maps into a heat-map (red = strong influence).

In [ ]:
def make_gradcam_model(model, layer_name="out_relu"):
    names = [n for n in ("image", "meta") if n in [l.name for l in model.layers]]
    inputs = {n: model.get_layer(n).output for n in names}
    inputs = inputs["image"] if names == ["image"] else inputs
    return keras.Model(inputs, [model.get_layer(layer_name).output, model.output])

def grad_cam(gc_model, inputs, class_idx=None):
    with tf.GradientTape() as tape:
        conv, preds = gc_model(inputs, training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(weights * conv, axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0].numpy()
    return cam, preds[0].numpy()

gc_image, gc_fusion = make_gradcam_model(image_model), make_gradcam_model(fusion_model)

picks = [te_idx[y_all[te_idx] == c][0] for c in range(NUM_CLASSES) if (y_all[te_idx] == c).any()]
fig, axes = plt.subplots(len(picks), 3, figsize=(11, 3.6 * len(picks)), squeeze=False)
for row, i in enumerate(picks):
    img = X_img[i].astype("float32")[None]
    cam_a, p_a = grad_cam(gc_image, img)
    cam_d, p_d = grad_cam(gc_fusion, {"image": img, "meta": X_meta[i][None]})
    axes[row, 0].imshow(X_img[i]); axes[row, 0].set_title(f"True: {CLASS_NAMES[y_all[i]]} - {CLASS_FULL[CLASS_NAMES[y_all[i]]]}", fontsize=9)
    for ax, cam, p, name in [(axes[row, 1], cam_a, p_a, "A: Image model"), (axes[row, 2], cam_d, p_d, "D: Fusion model")]:
        ax.imshow(X_img[i]); ax.imshow(cam, cmap="jet", alpha=0.4)
        ax.set_title(f"{name}: {CLASS_NAMES[p.argmax()]} ({p.max():.0%})", fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/gradcam_examples.png", dpi=100, bbox_inches="tight"); plt.show()

## 11. Predict one case (demo)

`predict_case(...)` runs the full pipeline on one photo + the clinical answers and shows what every model predicts,
with the fusion model's Grad-CAM. Symptoms accept `True` / `False` / `None` (unknown). The UI can call this same function.

In [ ]:
def to_flag(v):
    if v is None:
        return "UNK"
    if isinstance(v, str):
        return v.strip().upper()
    return "TRUE" if v else "FALSE"

def predict_case(image_path, age, region, itch, grew, hurt, changed, bleed, elevation, show=True):
    img = preprocess_photo(image_path).astype("float32")[None]
    row = pd.DataFrame([{"age": age, "region": str(region), "itch": to_flag(itch), "grew": to_flag(grew),
                         "hurt": to_flag(hurt), "changed": to_flag(changed), "bleed": to_flag(bleed),
                         "elevation": to_flag(elevation)}])
    meta_vec = encode_clinical(row).to_numpy("float32")
    p_img = image_model(img, training=False).numpy()[0]
    p_txt = text_model.predict_proba(meta_vec)[0]
    p_late = LATE_W * p_img + (1 - LATE_W) * p_txt
    cam, p_fus = grad_cam(gc_fusion, {"image": img, "meta": meta_vec})
    table = pd.DataFrame({"A: Image": p_img, "B: Text": p_txt, "C: Late fusion": p_late, "D: Feature fusion": p_fus},
                         index=[f"{c} - {CLASS_FULL[c]}" for c in CLASS_NAMES])
    if show:
        fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
        axes[0].imshow(img[0].astype("uint8")); axes[0].set_title("Normalised photo")
        axes[1].imshow(img[0].astype("uint8")); axes[1].imshow(cam, cmap="jet", alpha=0.4)
        axes[1].set_title(f"Fusion: {CLASS_NAMES[p_fus.argmax()]} ({p_fus.max():.0%})")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout(); plt.show()
        display(table.style.format("{:.1%}").highlight_max(axis=0, color="#c6efce"))
    return table

r = data.iloc[te_idx[0]]                     # a patient the models never saw
print("Ground truth:", r["diagnostic"], "| clinical:", r[["age", "region"] + SYMPTOMS].to_dict())
_ = predict_case(r["path"], age=r["age"], region=r["region"], itch=r["itch"], grew=r["grew"], hurt=r["hurt"],
                 changed=r["changed"], bleed=r["bleed"], elevation=r["elevation"])

## 12. Saved files (`MyDrive/skin_fusion/`)

`image_model.keras` (A), `text_model.joblib` (B), `fusion_model.keras` (D), `config.json` (normalisation targets,
late-fusion weight, feature order), plus comparison tables, per-class tables, confusion matrices, training curves,
Grad-CAM examples and test predictions.

In [ ]:
config = {
    "dataset": "PAD-UFES-20 (Kaggle mahdavi1202/skin-cancer)",
    "class_names": CLASS_NAMES, "class_full_names": CLASS_FULL,
    "img_size": IMG_SIZE, "work_size": WORK_SIZE,
    "color_normalisation_targets": {"brightness": T_BRIGHT, "contrast": T_CONTR, "saturation": T_SAT},
    "text_features": TEXT_FEATURES, "regions": REGIONS, "symptom_encoding": {"TRUE": 1, "FALSE": 0, "UNK": 2},
    "late_fusion_image_weight": LATE_W,
    "gradcam_layer": "out_relu",
    "models": {"image": "image_model.keras", "text": "text_model.joblib", "fusion": "fusion_model.keras"},
    "test_results": {m: {k: round(float(v), 4) for k, v in res.items()} for m, res in RESULTS.items()},
}
with open(f"{OUT_DIR}/config.json", "w") as fh:
    json.dump(config, fh, indent=2)
for name in sorted(os.listdir(OUT_DIR)):
    print(f"{name:42s} {os.path.getsize(os.path.join(OUT_DIR, name)) / 1e6:8.2f} MB")

### Notes for the Phase II report
* **Dataset:** PAD-UFES-20 — 6 classes, smartphone (clinical) photos. Phase I planned HAM10000 (7 dermoscopic classes);
  the switch is needed because only PAD-UFES-20 has the patient-history fields (itch, hurt, grew, changed, bleed, elevation).
* **Text-model numbers differ from the TEXT notebook** because that notebook reports 5-fold averages; here one fixed split is used for all models.
* **Small test set:** ~450 test photos and only a handful of melanomas, so per-class MEL numbers swing between runs.
* **No leakage:** patient-grouped split; normalisation targets, SMOTE, augmentation and feature scaling all use training patients only.